##### ARTI 560 - Computer Vision

## Image Classification with Vision Transformer (ViT) - Exercise

### Objective

In this exercise, you will test the pretrained Vision Transformer (ViT) model on 5 real-world images that you find online.

You will:

1. Download 5 images for different classes in [ImageNet](https://github.com/Waikato/wekaDeeplearning4j/blob/master/docs/user-guide/class-maps/IMAGENET.md).

2. Load the ImageNet class names from a [text file](https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt).

3. Use ViT to predict the class for each image.

4. Record whether the prediction was correct.

#### Important Note

For this exercise, you MUST use the following KerasHub components:

- [keras_hub.models.ViTImageClassifier](https://keras.io/keras_hub/api/models/vit/vit_image_classifier/)

- [keras_hub.models.ViTImageClassifierPreprocessor](https://keras.io/keras_hub/api/models/vit/vit_image_classifier_preprocessor/)

This ensures your input preprocessing (resizing + normalization) matches what the pretrained ViT model expects.

Do not replace the preprocessor with manual normalization (such as dividing by 255), because it may produce incorrect predictions.

In [30]:
# Golden Retriever
!wget https://upload.wikimedia.org/wikipedia/commons/6/6e/Golde33443.jpg -O /content/golden_retriever.jpg

--2026-03-05 01:37:55--  https://upload.wikimedia.org/wikipedia/commons/6/6e/Golde33443.jpg
Resolving upload.wikimedia.org (upload.wikimedia.org)... 208.80.153.240, 2620:0:860:ed1a::2:b
Connecting to upload.wikimedia.org (upload.wikimedia.org)|208.80.153.240|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 212652 (208K) [image/jpeg]
Saving to: ‘/content/golden_retriever.jpg’

/content/golden_ret 100%[===================>] 207.67K  --.-KB/s    in 0.09s   

2026-03-05 01:37:55 (2.27 MB/s) - ‘/content/golden_retriever.jpg’ saved [212652/212652]



In [31]:

# Giant Panda
!wget https://upload.wikimedia.org/wikipedia/commons/0/0f/Grosser_Panda.JPG -O /content/panda.jpg


--2026-03-05 01:37:57--  https://upload.wikimedia.org/wikipedia/commons/0/0f/Grosser_Panda.JPG
Resolving upload.wikimedia.org (upload.wikimedia.org)... 208.80.153.240, 2620:0:860:ed1a::2:b
Connecting to upload.wikimedia.org (upload.wikimedia.org)|208.80.153.240|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5012179 (4.8M) [image/jpeg]
Saving to: ‘/content/panda.jpg’

/content/panda.jpg  100%[===================>]   4.78M  17.2MB/s    in 0.3s    

2026-03-05 01:37:57 (17.2 MB/s) - ‘/content/panda.jpg’ saved [5012179/5012179]



In [32]:
# Pizza
!wget https://upload.wikimedia.org/wikipedia/commons/d/d3/Supreme_pizza.jpg -O /content/pizza.jpg

--2026-03-05 01:37:58--  https://upload.wikimedia.org/wikipedia/commons/d/d3/Supreme_pizza.jpg
Resolving upload.wikimedia.org (upload.wikimedia.org)... 208.80.153.240, 2620:0:860:ed1a::2:b
Connecting to upload.wikimedia.org (upload.wikimedia.org)|208.80.153.240|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1097672 (1.0M) [image/jpeg]
Saving to: ‘/content/pizza.jpg’

/content/pizza.jpg  100%[===================>]   1.05M  --.-KB/s    in 0.1s    

2026-03-05 01:37:58 (7.07 MB/s) - ‘/content/pizza.jpg’ saved [1097672/1097672]



In [33]:
# Tiger
!wget https://upload.wikimedia.org/wikipedia/commons/5/56/Tiger.50.jpg -O /content/tiger.jpg

--2026-03-05 01:37:59--  https://upload.wikimedia.org/wikipedia/commons/5/56/Tiger.50.jpg
Resolving upload.wikimedia.org (upload.wikimedia.org)... 208.80.153.240, 2620:0:860:ed1a::2:b
Connecting to upload.wikimedia.org (upload.wikimedia.org)|208.80.153.240|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 14603 (14K) [image/jpeg]
Saving to: ‘/content/tiger.jpg’

/content/tiger.jpg  100%[===================>]  14.26K  --.-KB/s    in 0s      

2026-03-05 01:38:00 (134 MB/s) - ‘/content/tiger.jpg’ saved [14603/14603]



In [35]:
# Jellyfish
!wget https://upload.wikimedia.org/wikipedia/commons/4/44/Jelly_cc11.jpg -O /content/jellyfish.jpg

--2026-03-05 01:38:08--  https://upload.wikimedia.org/wikipedia/commons/4/44/Jelly_cc11.jpg
Resolving upload.wikimedia.org (upload.wikimedia.org)... 208.80.153.240, 2620:0:860:ed1a::2:b
Connecting to upload.wikimedia.org (upload.wikimedia.org)|208.80.153.240|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 78824 (77K) [image/jpeg]
Saving to: ‘/content/jellyfish.jpg’

/content/jellyfish. 100%[===================>]  76.98K  --.-KB/s    in 0.05s   

2026-03-05 01:38:08 (1.43 MB/s) - ‘/content/jellyfish.jpg’ saved [78824/78824]



In [40]:
# Import required libraries
import numpy as np
from PIL import Image
import requests
import keras
import keras_hub

# Load the ViT preprocessor (handles resizing and normalization automatically)
vit_processor = keras_hub.models.ViTImageClassifierPreprocessor.from_preset(
    "vit_base_patch16_224_imagenet"
)

# Load the pretrained Vision Transformer model
vit_model = keras_hub.models.ViTImageClassifier.from_preset(
    "vit_base_patch16_224_imagenet",
    preprocessor=vit_processor
)

# Load ImageNet class labels
labels_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
imagenet_labels = requests.get(labels_url).text.splitlines()


# Define the test images and their expected classes
image_samples = [
    {"file": "/content/golden_retriever.jpg", "true": "golden retriever"},
    {"file": "/content/panda.jpg", "true": "giant panda"},
    {"file": "/content/pizza.jpg", "true": "pizza"},
    {"file": "/content/tiger.jpg", "true": "tiger"},
    {"file": "/content/jellyfish.jpg", "true": "jellyfish"},
]

print("===== Vision Transformer Classification Results =====")

for i, item in enumerate(image_samples, start=1):
    print(f"\n--- Image {i} ---")

    # Load image
    img = Image.open(item["file"]).convert("RGB")

    img_np = np.array(img, dtype=np.uint8)
    batch = np.expand_dims(img_np, axis=0)

    # Predict
    logits = vit_model.predict(batch, verbose=0)
    probs = keras.ops.softmax(logits, axis=-1)
    probs = np.array(probs)[0]

    pred_id = int(np.argmax(probs))
    pred_label = imagenet_labels[pred_id]
    conf = float(probs[pred_id])

    correct = "Yes" if item["true"].lower() in pred_label.lower() else "No"

    print("Image File      :", item["file"])
    print("Predicted Label :", pred_label, f"({conf:.2%})")
    print("True Label      :", item["true"])
    print("Correct?        :", correct)

===== Vision Transformer Classification Results =====

--- Image 1 ---
Image File      : /content/golden_retriever.jpg
Predicted Label : golden retriever (94.08%)
True Label      : golden retriever
Correct?        : Yes

--- Image 2 ---
Image File      : /content/panda.jpg
Predicted Label : giant panda (99.73%)
True Label      : giant panda
Correct?        : Yes

--- Image 3 ---
Image File      : /content/pizza.jpg
Predicted Label : pizza (98.68%)
True Label      : pizza
Correct?        : Yes

--- Image 4 ---
Image File      : /content/tiger.jpg
Predicted Label : tiger (92.99%)
True Label      : tiger
Correct?        : Yes

--- Image 5 ---
Image File      : /content/jellyfish.jpg
Predicted Label : jellyfish (99.95%)
True Label      : jellyfish
Correct?        : Yes


### Record Your Results

Fill the table below based on your results:

| Image File            | Predicted Label           | True Label (What you searched) | Correct? (Yes/No) |
|-----------------------|---------------------------|---------------------------------|-------------------|
| golden_retriever.jpg  | golden retriever (94.08%) | golden retriever                | Yes               |
| panda.jpg             | giant panda (99.73%)      | giant panda                     | Yes               |
| pizza.jpg             | pizza (98.68%)            | pizza                           | Yes               |
| tiger.jpg             | tiger (92.99%)            | tiger                           | Yes               |
| jellyfish.jpg         | jellyfish (99.95%)        | jellyfish                       | Yes               |